# Telecom Customer Segmentation using K-Means Clustering

---

This notebook performs **unsupervised customer segmentation** on the Telecom Customer Churn dataset. Rather than predicting churn directly, the goal is to discover natural groupings of customers based on their service usage patterns, financial behavior, and account characteristics, without using the `Churn` label in any step of the modeling.

Understanding these segments enables the business to:
- Design targeted retention and upselling strategies per customer group.
- Identify structurally high-risk segments by cross-referencing cluster profiles post-hoc.
- Optimize resource allocation for customer success teams.

**Algorithm of Choice: K-Means Clustering**

K-Means is selected for this task because:
- It produces hard, interpretable cluster assignments.
- It scales efficiently to this dataset size (7,043 records).
- When combined with proper preprocessing (encoding + scaling) and dimensionality reduction (PCA), it delivers robust and meaningful segments from mixed-type tabular data.

---

**Pipeline Overview:**

1. Library Installation and Import  
2. Data Loading  
3. Exploratory Data Analysis (EDA)  
4. Data Cleaning  
5. Feature Engineering and Preprocessing  
6. Optimal K Selection (Elbow + Silhouette + Davies-Bouldin + Calinski-Harabasz)  
7. K-Means Modeling  
8. Dimensionality Reduction and Visualization (PCA)  
9. Cluster Profiling and Business Interpretation  
10. Model and Artifact Persistence  
11. Evaluation Summary

---

## 1. Library Installation and Import

All required libraries are installed inline to ensure full reproducibility across environments. The core libraries used are:

- **pandas / numpy**: Data manipulation and numerical computation.
- **scikit-learn**: Preprocessing pipelines, K-Means implementation, and evaluation metrics (Silhouette Score, Davies-Bouldin Index, Calinski-Harabasz Score).
- **matplotlib / seaborn**: Static visualizations and heatmaps.
- **plotly**: Interactive cluster visualizations.
- **yellowbrick**: Visual diagnostic tools for machine learning, including the KElbowVisualizer and SilhouetteVisualizer.
- **joblib**: Model serialization for artifact persistence.

In [ ]:
import subprocess
import sys

packages = [
    'pandas', 'numpy', 'scikit-learn', 'matplotlib',
    'seaborn', 'plotly', 'yellowbrick', 'joblib', 'kaleido'
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('All libraries installed successfully.')

In [ ]:
import warnings
import os
import json
import joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score
)
from yellowbrick.cluster import KElbowVisualizer, SilhouetteVisualizer

warnings.filterwarnings('ignore')
np.random.seed(42)

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

KNOWLEDGE_DIR = 'clustering_knowledge'
os.makedirs(KNOWLEDGE_DIR, exist_ok=True)

import sklearn
print(f'scikit-learn : {sklearn.__version__}')
print(f'pandas       : {pd.__version__}')
print(f'numpy        : {np.__version__}')
print(f'Knowledge dir: {os.path.abspath(KNOWLEDGE_DIR)}')

---

## 2. Data Loading

The dataset is loaded from the `../Dataset/` directory relative to the Model folder. The `Churn` column is intentionally **retained in the DataFrame for post-hoc analysis only** and is strictly excluded from all preprocessing and modeling steps.

In [ ]:
DATA_PATH = os.path.join('..', 'Dataset', 'IT_customer_churn.csv')

df_raw = pd.read_csv(DATA_PATH)

print(f'Dataset shape    : {df_raw.shape}')
print(f'Total records    : {df_raw.shape[0]:,}')
print(f'Total features   : {df_raw.shape[1]}')
print('\nColumn list:')
print(df_raw.columns.tolist())

In [ ]:
df_raw.head(10)

---

## 3. Exploratory Data Analysis (EDA)

Before modeling, a structured EDA is performed to understand:
- Data types and cardinality of each feature.
- Missing values and data quality issues.
- Statistical distributions of numerical features.
- Frequency distributions of categorical features.

This step informs every subsequent preprocessing decision.

In [ ]:
print('Data Types and Non-Null Counts:')
print('=' * 50)
df_raw.info()

In [ ]:
print('Descriptive Statistics for Numerical Features:')
df_raw[['tenure', 'MonthlyCharges', 'TotalCharges']].describe().round(2)

In [ ]:
print('Missing Values per Column:')
missing = df_raw.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No null values detected.')

blank_total = (df_raw['TotalCharges'].astype(str).str.strip() == '').sum()
print(f'\nBlank strings in TotalCharges (whitespace-only): {blank_total}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Distribution of Numerical Features', fontsize=15, fontweight='bold', y=1.02)

numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
palette = ['#4C72B0', '#DD8452', '#55A868']

for ax, col, color in zip(axes, numerical_cols, palette):
    data = pd.to_numeric(df_raw[col], errors='coerce').dropna()
    ax.hist(data, bins=40, color=color, alpha=0.85, edgecolor='white')
    ax.axvline(data.mean(), color='red', linestyle='--', linewidth=1.5, label=f'Mean: {data.mean():.1f}')
    ax.axvline(data.median(), color='gold', linestyle='-.', linewidth=1.5, label=f'Median: {data.median():.1f}')
    ax.set_title(col, fontsize=13, fontweight='bold')
    ax.set_xlabel('Value', fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(KNOWLEDGE_DIR, 'eda_numerical_distributions.png'), bbox_inches='tight')
plt.show()
print('Saved: eda_numerical_distributions.png')

In [ ]:
categorical_cols = ['InternetService', 'Contract', 'PaymentMethod',
                    'TechSupport', 'OnlineSecurity', 'StreamingTV']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Frequency Distribution of Key Categorical Features', fontsize=15, fontweight='bold', y=1.01)

for ax, col in zip(axes.flatten(), categorical_cols):
    counts = df_raw[col].value_counts()
    bars = ax.barh(counts.index, counts.values,
                   color=sns.color_palette('muted', len(counts)), edgecolor='white')
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.set_xlabel('Count', fontsize=10)
    for bar, val in zip(bars, counts.values):
        ax.text(val + 30, bar.get_y() + bar.get_height() / 2,
                f'{val:,}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(KNOWLEDGE_DIR, 'eda_categorical_distributions.png'), bbox_inches='tight')
plt.show()
print('Saved: eda_categorical_distributions.png')

In [ ]:
df_temp = df_raw.copy()
df_temp['TotalCharges'] = pd.to_numeric(df_temp['TotalCharges'], errors='coerce')

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Bivariate: Monthly Charges vs Tenure by Key Categorical Features',
             fontsize=14, fontweight='bold')

group_cols = ['InternetService', 'Contract', 'PaymentMethod']
for ax, col in zip(axes, group_cols):
    categories = df_temp[col].unique()
    palette_map = dict(zip(categories, sns.color_palette('tab10', len(categories))))
    for cat in categories:
        subset = df_temp[df_temp[col] == cat]
        ax.scatter(subset['tenure'], subset['MonthlyCharges'],
                   alpha=0.25, s=12, label=cat, color=palette_map[cat])
    ax.set_xlabel('Tenure (months)', fontsize=10)
    ax.set_ylabel('Monthly Charges ($)', fontsize=10)
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.legend(fontsize=7, loc='best')

plt.tight_layout()
plt.savefig(os.path.join(KNOWLEDGE_DIR, 'eda_bivariate_scatter.png'), bbox_inches='tight')
plt.show()
print('Saved: eda_bivariate_scatter.png')

---

## 4. Data Cleaning

The following cleaning steps are applied:

**Issue 1 - TotalCharges as String with Blank Values:**  
`TotalCharges` is stored as a string due to 11 records containing only whitespace. These correspond to customers with `tenure = 0` (brand-new subscribers who have not yet been billed). The column is coerced to float, and blank entries are imputed with `0.0`, which is a semantically correct imputation since no charges have accrued yet.

**Issue 2 - Churn Column Exclusion:**  
The `Churn` column is dropped from the modeling DataFrame. It is retained in a separate Series for post-hoc cluster validation only.

**Issue 3 - Gender Column:**  
`gender` is excluded from the clustering feature set. Gender does not reflect service behavior or spending patterns and its inclusion could introduce demographic bias without adding predictive signal for segmentation.

In [ ]:
df = df_raw.copy()

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0.0)

churn_series = df['Churn'].copy()

FEATURES_TO_DROP = ['Churn', 'gender']
df_model = df.drop(columns=FEATURES_TO_DROP)

print(f'Modeling DataFrame shape : {df_model.shape}')
print(f'TotalCharges null count  : {df_model["TotalCharges"].isnull().sum()}')
print(f'Churn series length      : {len(churn_series)}')
print('\nFeatures used for clustering:')
print(df_model.columns.tolist())

---

## 5. Feature Engineering and Preprocessing

K-Means operates on Euclidean distance and requires all features to be:
1. **Numeric** - categorical features must be encoded.
2. **On the same scale** - numerical features must be standardized.

**Encoding Strategy for Categorical Features:**

- **Binary features** (`Yes`/`No`): Mapped to 0/1 directly.
- **Multi-level service features** (e.g., `OnlineSecurity`: No / No internet service / Yes): Mapped to an ordinal scale (0 = No internet service, 1 = No, 2 = Yes) to preserve the natural ordering of service provisioning.
- **`InternetService`** (DSL / Fiber optic / No): One-Hot Encoded to avoid imposing false ordinal relationships.
- **`Contract`**: Ordinal encoded (0 = Month-to-month, 1 = One year, 2 = Two year) to reflect increasing commitment level.
- **`PaymentMethod`**: One-Hot Encoded.

**Scaling:**  
All encoded features are passed through `StandardScaler` to zero-center and unit-normalize distributions, ensuring no single feature dominates distance calculations.

In [ ]:
df_encoded = df_model.copy()

binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in binary_cols:
    df_encoded[col] = df_encoded[col].map({'Yes': 1, 'No': 0})

service_map = {'No internet service': 0, 'No phone service': 0, 'No': 1, 'Yes': 2}
service_cols = [
    'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'
]
for col in service_cols:
    df_encoded[col] = df_encoded[col].map(service_map)

df_encoded['Contract'] = df_encoded['Contract'].map(
    {'Month-to-month': 0, 'One year': 1, 'Two year': 2}
)

internet_dummies = pd.get_dummies(df_encoded['InternetService'],
                                  prefix='Internet', drop_first=False).astype(int)
payment_dummies  = pd.get_dummies(df_encoded['PaymentMethod'],
                                  prefix='Payment', drop_first=False).astype(int)

df_encoded = df_encoded.drop(columns=['InternetService', 'PaymentMethod'])
df_encoded = pd.concat([df_encoded, internet_dummies, payment_dummies], axis=1)

print(f'Encoded DataFrame shape : {df_encoded.shape}')
print('\nAll feature columns:')
print(df_encoded.columns.tolist())
print('\nNull count:', df_encoded.isnull().sum().sum())

In [ ]:
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(df_encoded)
X_scaled_df = pd.DataFrame(X_scaled, columns=df_encoded.columns)

print(f'Scaled matrix shape     : {X_scaled.shape}')
print(f'Mean (should be ~0)     : {X_scaled.mean():.6f}')
print(f'Std Dev (should be ~1)  : {X_scaled.std():.6f}')

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
corr = X_scaled_df.corr()['tenure'].drop('tenure').sort_values()
colors = ['#d73027' if v < 0 else '#4575b4' for v in corr.values]
ax.barh(corr.index, corr.values, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Feature Correlation with Tenure (after scaling)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Pearson Correlation', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(KNOWLEDGE_DIR, 'feature_correlation_tenure.png'), bbox_inches='tight')
plt.show()
print('Saved: feature_correlation_tenure.png')

---

## 6. Optimal K Selection

Selecting the optimal number of clusters `K` is the most critical decision in K-Means. Four complementary methods are used to arrive at a well-justified choice:

**Method 1 - Elbow Method (Inertia / WCSS):**  
Plots the Within-Cluster Sum of Squares as a function of K. The optimal K is the elbow point where the rate of decrease sharply diminishes, indicating diminishing returns on adding more clusters.

**Method 2 - Silhouette Score:**  
Measures how well each point fits its assigned cluster compared to the nearest neighboring cluster. Score ranges from -1 to +1; a higher score indicates better-defined, more separated clusters.

**Method 3 - Davies-Bouldin Index:**  
Computes the average ratio of within-cluster scatter to between-cluster separation. A lower value indicates better cluster compactness and separation.

**Method 4 - Calinski-Harabasz Score:**  
The ratio of between-cluster dispersion to within-cluster dispersion. A higher value indicates better-defined clusters.

In [ ]:
K_RANGE = range(2, 11)

inertia_values            = []
silhouette_values         = []
davies_bouldin_values     = []
calinski_harabasz_values  = []

print('Computing cluster metrics for K = 2 to 10 ...')
for k in K_RANGE:
    km = KMeans(n_clusters=k, init='k-means++', n_init=20, max_iter=500, random_state=42)
    labels = km.fit_predict(X_scaled)
    inertia_values.append(km.inertia_)
    silhouette_values.append(
        silhouette_score(X_scaled, labels, sample_size=3000, random_state=42)
    )
    davies_bouldin_values.append(davies_bouldin_score(X_scaled, labels))
    calinski_harabasz_values.append(calinski_harabasz_score(X_scaled, labels))
    print(
        f'  K={k:2d} | Inertia: {km.inertia_:,.1f} | '
        f'Silhouette: {silhouette_values[-1]:.4f} | '
        f'DB: {davies_bouldin_values[-1]:.4f} | '
        f'CH: {calinski_harabasz_values[-1]:.1f}'
    )

print('\nMetric computation complete.')

In [ ]:
k_list = list(K_RANGE)

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('K Selection Diagnostic Metrics', fontsize=16, fontweight='bold', y=1.01)

axes[0, 0].plot(k_list, inertia_values, marker='o', color='#4C72B0', linewidth=2.5, markersize=7)
axes[0, 0].set_title('Elbow Method (Inertia / WCSS)', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('Number of Clusters (K)')
axes[0, 0].set_ylabel('Inertia')
axes[0, 0].set_xticks(k_list)

axes[0, 1].plot(k_list, silhouette_values, marker='s', color='#55A868', linewidth=2.5, markersize=7)
axes[0, 1].set_title('Silhouette Score (Higher is Better)', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('Number of Clusters (K)')
axes[0, 1].set_ylabel('Silhouette Score')
axes[0, 1].set_xticks(k_list)
best_sil_k = k_list[int(np.argmax(silhouette_values))]
axes[0, 1].axvline(best_sil_k, color='red', linestyle='--', alpha=0.7, label=f'Best K={best_sil_k}')
axes[0, 1].legend()

axes[1, 0].plot(k_list, davies_bouldin_values, marker='^', color='#DD8452', linewidth=2.5, markersize=7)
axes[1, 0].set_title('Davies-Bouldin Index (Lower is Better)', fontsize=13, fontweight='bold')
axes[1, 0].set_xlabel('Number of Clusters (K)')
axes[1, 0].set_ylabel('DB Index')
axes[1, 0].set_xticks(k_list)
best_db_k = k_list[int(np.argmin(davies_bouldin_values))]
axes[1, 0].axvline(best_db_k, color='red', linestyle='--', alpha=0.7, label=f'Best K={best_db_k}')
axes[1, 0].legend()

axes[1, 1].plot(k_list, calinski_harabasz_values, marker='D', color='#8172B2', linewidth=2.5, markersize=7)
axes[1, 1].set_title('Calinski-Harabasz Score (Higher is Better)', fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel('Number of Clusters (K)')
axes[1, 1].set_ylabel('CH Score')
axes[1, 1].set_xticks(k_list)
best_ch_k = k_list[int(np.argmax(calinski_harabasz_values))]
axes[1, 1].axvline(best_ch_k, color='red', linestyle='--', alpha=0.7, label=f'Best K={best_ch_k}')
axes[1, 1].legend()

plt.tight_layout()
plt.savefig(os.path.join(KNOWLEDGE_DIR, 'k_selection_metrics.png'), bbox_inches='tight')
plt.show()
print('Saved: k_selection_metrics.png')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
visualizer = KElbowVisualizer(
    KMeans(init='k-means++', n_init=20, max_iter=500, random_state=42),
    k=(2, 11), metric='distortion', timings=False, ax=ax
)
visualizer.fit(X_scaled)
visualizer.finalize()
ax.set_title('Yellowbrick KElbow Visualizer - Optimal K Detection',
             fontsize=13, fontweight='bold')
plt.savefig(os.path.join(KNOWLEDGE_DIR, 'kelbow_yellowbrick.png'), bbox_inches='tight')
plt.show()

elbow_k = visualizer.elbow_value_
print(f'Yellowbrick detected elbow at K = {elbow_k}')

In [ ]:
metrics_df = pd.DataFrame({
    'K': k_list,
    'Inertia': inertia_values,
    'Silhouette_Score': silhouette_values,
    'Davies_Bouldin_Index': davies_bouldin_values,
    'Calinski_Harabasz_Score': calinski_harabasz_values
})

print('Full Metric Table:')
print(metrics_df.to_string(index=False))
print(f'\nBest K by Silhouette Score  : {k_list[int(np.argmax(silhouette_values))]}')
print(f'Best K by Davies-Bouldin    : {k_list[int(np.argmin(davies_bouldin_values))]}')
print(f'Best K by Calinski-Harabasz : {k_list[int(np.argmax(calinski_harabasz_values))]}')
print(f'Best K by Yellowbrick Elbow : {elbow_k}')

---

## 7. K-Means Modeling

Based on the convergence of diagnostic metrics above, the optimal number of clusters **K = 4** is selected. This value represents the best balance between:
- Minimizing inertia (compact clusters).
- Maximizing silhouette score (well-separated clusters).
- Business interpretability (4 distinct customer personas are directly actionable).

**Configuration:**
- `init='k-means++'`: Smarter centroid initialization that reduces convergence time and avoids poor local minima.
- `n_init=50`: The algorithm is run 50 times with different centroid seeds; the best result (lowest inertia) is retained.
- `max_iter=500`: Maximum iterations per single run.
- `random_state=42`: Ensures full reproducibility.

In [ ]:
OPTIMAL_K = 4

kmeans_final = KMeans(
    n_clusters=OPTIMAL_K,
    init='k-means++',
    n_init=50,
    max_iter=500,
    random_state=42
)

cluster_labels = kmeans_final.fit_predict(X_scaled)

df['Cluster']       = cluster_labels
df_model['Cluster'] = cluster_labels

print(f'K-Means converged in {kmeans_final.n_iter_} iterations.')
print(f'Final Inertia (WCSS) : {kmeans_final.inertia_:,.2f}')
print('\nCluster Size Distribution:')
cluster_counts = pd.Series(cluster_labels).value_counts().sort_index()
for k, count in cluster_counts.items():
    pct = count / len(cluster_labels) * 100
    print(f'  Cluster {k}: {count:,} customers ({pct:.1f}%)')

In [ ]:
final_silhouette = silhouette_score(X_scaled, cluster_labels, sample_size=5000, random_state=42)
final_db_score   = davies_bouldin_score(X_scaled, cluster_labels)
final_ch_score   = calinski_harabasz_score(X_scaled, cluster_labels)

print('Final Model Evaluation Metrics (K=4):')
print('=' * 45)
print(f'  Silhouette Score         : {final_silhouette:.4f}')
print(f'  Davies-Bouldin Index     : {final_db_score:.4f}')
print(f'  Calinski-Harabasz Score  : {final_ch_score:.2f}')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
sil_viz = SilhouetteVisualizer(
    KMeans(n_clusters=OPTIMAL_K, init='k-means++', n_init=50, max_iter=500, random_state=42),
    colors='yellowbrick', ax=ax
)
sil_viz.fit(X_scaled)
sil_viz.finalize()
ax.set_title(f'Silhouette Plot - K-Means with K={OPTIMAL_K}', fontsize=13, fontweight='bold')
plt.savefig(os.path.join(KNOWLEDGE_DIR, 'silhouette_plot_k4.png'), bbox_inches='tight')
plt.show()
print('Saved: silhouette_plot_k4.png')

---

## 8. Dimensionality Reduction and Visualization (PCA)

Since the feature space has high dimensionality after encoding, **Principal Component Analysis (PCA)** is used to project the data into 2D and 3D spaces for visualization. PCA identifies the directions of maximum variance and projects each sample onto those axes.

The PCA projection is used **solely for visualization**. The K-Means model was trained and evaluated on the full-dimensional scaled feature space.

In [ ]:
pca_full = PCA(random_state=42)
pca_full.fit(X_scaled)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_components_95 = int(np.argmax(cumvar >= 0.95)) + 1

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].bar(
    range(1, min(16, len(pca_full.explained_variance_ratio_)) + 1),
    pca_full.explained_variance_ratio_[:15] * 100,
    color='#4C72B0', edgecolor='white'
)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance (%)')
axes[0].set_title('Scree Plot - Individual Explained Variance', fontsize=13, fontweight='bold')

axes[1].plot(range(1, len(cumvar) + 1), cumvar * 100, color='#55A868', linewidth=2.5)
axes[1].axhline(95, color='red', linestyle='--', linewidth=1.5, label='95% threshold')
axes[1].axvline(n_components_95, color='orange', linestyle='--',
                linewidth=1.5, label=f'{n_components_95} components')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance (%)')
axes[1].set_title('Cumulative Explained Variance', fontsize=13, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(KNOWLEDGE_DIR, 'pca_variance_explained.png'), bbox_inches='tight')
plt.show()
print(f'Components needed for 95% variance: {n_components_95}')

In [ ]:
pca_2d    = PCA(n_components=2, random_state=42)
X_pca_2d  = pca_2d.fit_transform(X_scaled)

pca_2d_df = pd.DataFrame(X_pca_2d, columns=['PC1', 'PC2'])
pca_2d_df['Cluster'] = cluster_labels
pca_2d_df['Churn']   = churn_series.values

var_pc1 = pca_2d.explained_variance_ratio_[0] * 100
var_pc2 = pca_2d.explained_variance_ratio_[1] * 100

print(f'PC1 explained variance: {var_pc1:.2f}%')
print(f'PC2 explained variance: {var_pc2:.2f}%')
print(f'Total (2D)            : {var_pc1 + var_pc2:.2f}%')

In [ ]:
CLUSTER_COLORS = ['#E63946', '#457B9D', '#2A9D8F', '#E9C46A']

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('K-Means Clustering - PCA 2D Projection', fontsize=16, fontweight='bold')

for cid in range(OPTIMAL_K):
    mask = pca_2d_df['Cluster'] == cid
    axes[0].scatter(
        pca_2d_df.loc[mask, 'PC1'], pca_2d_df.loc[mask, 'PC2'],
        s=12, alpha=0.45, color=CLUSTER_COLORS[cid], label=f'Cluster {cid}'
    )

centroids_2d = pca_2d.transform(kmeans_final.cluster_centers_)
axes[0].scatter(centroids_2d[:, 0], centroids_2d[:, 1],
                s=250, marker='X', color='black', zorder=10, label='Centroids')
axes[0].set_xlabel(f'PC1 ({var_pc1:.1f}% variance)', fontsize=11)
axes[0].set_ylabel(f'PC2 ({var_pc2:.1f}% variance)', fontsize=11)
axes[0].set_title('Colored by Cluster Assignment', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=9, markerscale=2)

churn_color_map = {'Yes': '#E63946', 'No': '#457B9D'}
for churn_val in ['No', 'Yes']:
    mask = pca_2d_df['Churn'] == churn_val
    axes[1].scatter(
        pca_2d_df.loc[mask, 'PC1'], pca_2d_df.loc[mask, 'PC2'],
        s=10, alpha=0.35, color=churn_color_map[churn_val], label=f'Churn = {churn_val}'
    )
axes[1].set_xlabel(f'PC1 ({var_pc1:.1f}% variance)', fontsize=11)
axes[1].set_ylabel(f'PC2 ({var_pc2:.1f}% variance)', fontsize=11)
axes[1].set_title('Colored by Churn Label (Post-hoc)', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10, markerscale=2)

plt.tight_layout()
plt.savefig(os.path.join(KNOWLEDGE_DIR, 'pca_2d_cluster_visualization.png'), bbox_inches='tight')
plt.show()
print('Saved: pca_2d_cluster_visualization.png')

In [ ]:
pca_3d   = PCA(n_components=3, random_state=42)
X_pca_3d = pca_3d.fit_transform(X_scaled)

pca_3d_df = pd.DataFrame(X_pca_3d, columns=['PC1', 'PC2', 'PC3'])
pca_3d_df['Cluster'] = cluster_labels.astype(str)

fig_3d = px.scatter_3d(
    pca_3d_df, x='PC1', y='PC2', z='PC3', color='Cluster',
    color_discrete_sequence=CLUSTER_COLORS,
    opacity=0.5,
    title='K-Means Clustering - Interactive 3D PCA Projection',
    labels={
        'PC1': f"PC1 ({pca_3d.explained_variance_ratio_[0]*100:.1f}%)",
        'PC2': f"PC2 ({pca_3d.explained_variance_ratio_[1]*100:.1f}%)",
        'PC3': f"PC3 ({pca_3d.explained_variance_ratio_[2]*100:.1f}%)"
    }
)
fig_3d.update_traces(marker=dict(size=3))
fig_3d.update_layout(title_font_size=15, title_x=0.5, legend_title_text='Cluster')
fig_3d.write_html(os.path.join(KNOWLEDGE_DIR, 'pca_3d_interactive.html'))
fig_3d.show()
print('Saved: pca_3d_interactive.html')

---

## 9. Cluster Profiling and Business Interpretation

The most valuable output of a segmentation model is a rich, interpretable description of each cluster. The profiles below are constructed by computing the mean (for numerical features) and mode with proportion (for categorical features) of each cluster, enabling the derivation of actionable customer personas.

The actual `Churn` label (which was never used in modeling) is overlaid post-hoc to understand each segment's natural churn propensity. This is a critical validation step that confirms the clustering has discovered structurally meaningful, rather than arbitrary, segments.

In [ ]:
df_profile = df.copy()
df_profile['TotalCharges'] = pd.to_numeric(df_profile['TotalCharges'], errors='coerce').fillna(0.0)

numerical_profile = (
    df_profile.groupby('Cluster')[['tenure', 'MonthlyCharges', 'TotalCharges']]
    .mean().round(2)
)
print('Cluster Means - Numerical Features:')
print(numerical_profile.to_string())

In [ ]:
cat_profile_cols = [
    'InternetService', 'Contract', 'PaymentMethod',
    'TechSupport', 'OnlineSecurity', 'StreamingTV',
    'StreamingMovies', 'MultipleLines', 'Partner', 'Dependents'
]

print('Cluster Profiles - Categorical Feature Mode:')
print('=' * 60)
for cid in range(OPTIMAL_K):
    subset = df_profile[df_profile['Cluster'] == cid]
    print(f'\nCluster {cid} (n={len(subset):,}):')
    for col in cat_profile_cols:
        mode_val = subset[col].mode().iloc[0]
        pct = (subset[col] == mode_val).mean() * 100
        print(f'  {col:<22}: {mode_val:<30} ({pct:.1f}%)')

In [ ]:
churn_by_cluster = df_profile.groupby('Cluster')['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100
).round(2)
churn_by_cluster.name = 'Churn_Rate_Pct'

size_by_cluster = df_profile.groupby('Cluster').size()
size_by_cluster.name = 'Cluster_Size'

cluster_summary = pd.concat([numerical_profile, churn_by_cluster, size_by_cluster], axis=1)

print('Full Cluster Summary Table:')
print(cluster_summary.to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Cluster Profiles - Key Metric Comparison', fontsize=16, fontweight='bold')

cluster_ids = [f'Cluster {i}' for i in range(OPTIMAL_K)]

axes[0, 0].bar(cluster_ids, cluster_summary['tenure'],
               color=CLUSTER_COLORS, edgecolor='white', width=0.55)
for i, v in enumerate(cluster_summary['tenure']):
    axes[0, 0].text(i, v + 0.5, f'{v:.1f}m', ha='center', va='bottom',
                    fontsize=10, fontweight='bold')
axes[0, 0].set_title('Average Tenure (months)', fontsize=13, fontweight='bold')
axes[0, 0].set_ylabel('Months')

axes[0, 1].bar(cluster_ids, cluster_summary['MonthlyCharges'],
               color=CLUSTER_COLORS, edgecolor='white', width=0.55)
for i, v in enumerate(cluster_summary['MonthlyCharges']):
    axes[0, 1].text(i, v + 0.5, f'${v:.1f}', ha='center', va='bottom',
                    fontsize=10, fontweight='bold')
axes[0, 1].set_title('Average Monthly Charges ($)', fontsize=13, fontweight='bold')
axes[0, 1].set_ylabel('USD')

axes[1, 0].bar(cluster_ids, cluster_summary['Churn_Rate_Pct'],
               color=CLUSTER_COLORS, edgecolor='white', width=0.55)
for i, v in enumerate(cluster_summary['Churn_Rate_Pct']):
    axes[1, 0].text(i, v + 0.3, f'{v:.1f}%', ha='center', va='bottom',
                    fontsize=10, fontweight='bold')
axes[1, 0].set_title('Churn Rate (Post-hoc Validation) %', fontsize=13, fontweight='bold')
axes[1, 0].set_ylabel('Churn Rate (%)')
overall_churn_pct = churn_series.map({'Yes': 1, 'No': 0}).mean() * 100
axes[1, 0].axhline(overall_churn_pct, color='black', linestyle='--',
                   linewidth=1.5, label='Overall Churn Rate')
axes[1, 0].legend()

axes[1, 1].bar(cluster_ids, cluster_summary['Cluster_Size'],
               color=CLUSTER_COLORS, edgecolor='white', width=0.55)
for i, v in enumerate(cluster_summary['Cluster_Size']):
    pct = v / len(df_profile) * 100
    axes[1, 1].text(i, v + 20, f'{v:,}\n({pct:.1f}%)', ha='center', va='bottom',
                    fontsize=10, fontweight='bold')
axes[1, 1].set_title('Cluster Size (Number of Customers)', fontsize=13, fontweight='bold')
axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.savefig(os.path.join(KNOWLEDGE_DIR, 'cluster_profile_bar_charts.png'), bbox_inches='tight')
plt.show()
print('Saved: cluster_profile_bar_charts.png')

In [ ]:
num_hm_cols = [
    'tenure', 'MonthlyCharges', 'TotalCharges',
    'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling'
]
df_hm = df_profile.copy()
df_hm['TotalCharges'] = pd.to_numeric(df_hm['TotalCharges'], errors='coerce').fillna(0)
for col in ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']:
    df_hm[col] = df_hm[col].map({'Yes': 1, 'No': 0})

heatmap_data = df_hm.groupby('Cluster')[num_hm_cols].mean()
heatmap_norm = (heatmap_data - heatmap_data.min()) / (heatmap_data.max() - heatmap_data.min())

fig, ax = plt.subplots(figsize=(13, 5))
sns.heatmap(
    heatmap_norm.T,
    annot=heatmap_data.T.round(2),
    fmt='.2f', cmap='RdYlGn',
    linewidths=0.5, linecolor='white', ax=ax,
    cbar_kws={'label': 'Normalized Value (0-1)'}
)
ax.set_title('Cluster Heatmap - Normalized Feature Means\n(Annotated with actual means)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Cluster', fontsize=11)
ax.set_ylabel('Feature', fontsize=11)
ax.set_xticklabels([f'Cluster {i}' for i in range(OPTIMAL_K)], rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(KNOWLEDGE_DIR, 'cluster_heatmap.png'), bbox_inches='tight')
plt.show()
print('Saved: cluster_heatmap.png')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Internet Service and Contract Type Distribution per Cluster',
             fontsize=15, fontweight='bold')

for cid, ax in enumerate(axes.flatten()):
    subset = df_profile[df_profile['Cluster'] == cid]
    internet_counts = subset['InternetService'].value_counts()
    bars = ax.bar(internet_counts.index, internet_counts.values,
                  color=sns.color_palette('muted', len(internet_counts)), edgecolor='white')
    churn_rate = (subset['Churn'] == 'Yes').mean() * 100
    ax.set_title(f'Cluster {cid} (n={len(subset):,}, Churn={churn_rate:.1f}%)',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Internet Service Type', fontsize=10)
    ax.set_ylabel('Count', fontsize=10)
    for bar, val in zip(bars, internet_counts.values):
        pct = val / len(subset) * 100
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10,
                f'{pct:.1f}%', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(KNOWLEDGE_DIR, 'cluster_internet_distribution.png'), bbox_inches='tight')
plt.show()
print('Saved: cluster_internet_distribution.png')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Numerical Feature Distributions per Cluster (Violin Plot)',
             fontsize=15, fontweight='bold')

for ax, col in zip(axes, ['tenure', 'MonthlyCharges', 'TotalCharges']):
    data_v = df_profile[['Cluster', col]].copy()
    data_v[col] = pd.to_numeric(data_v[col], errors='coerce').fillna(0)
    cluster_data = [data_v[data_v['Cluster'] == k][col].values for k in range(OPTIMAL_K)]
    parts = ax.violinplot(cluster_data, positions=range(OPTIMAL_K),
                          showmedians=True, showmeans=False)
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(CLUSTER_COLORS[i])
        pc.set_alpha(0.75)
    parts['cmedians'].set_color('white')
    parts['cbars'].set_color('black')
    parts['cmins'].set_color('black')
    parts['cmaxes'].set_color('black')
    ax.set_xticks(range(OPTIMAL_K))
    ax.set_xticklabels([f'C{k}' for k in range(OPTIMAL_K)], fontsize=10)
    ax.set_title(col, fontsize=13, fontweight='bold')
    ax.set_ylabel('Value', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(KNOWLEDGE_DIR, 'cluster_violin_plots.png'), bbox_inches='tight')
plt.show()
print('Saved: cluster_violin_plots.png')

In [ ]:
persona_notes = {}

print('Customer Segment Summary')
print('=' * 65)
for cid in range(OPTIMAL_K):
    subset       = df_profile[df_profile['Cluster'] == cid]
    internet_mode = subset['InternetService'].mode().iloc[0]
    contract_mode = subset['Contract'].mode().iloc[0]
    avg_tenure    = subset['tenure'].mean()
    avg_monthly   = subset['MonthlyCharges'].mean()
    churn_rate    = (subset['Churn'] == 'Yes').mean() * 100
    size          = len(subset)
    print(f'\nCluster {cid}')
    print(f'  Size           : {size:,} customers ({size/len(df_profile)*100:.1f}%)')
    print(f'  Avg Tenure     : {avg_tenure:.1f} months')
    print(f'  Avg Monthly $  : ${avg_monthly:.2f}')
    print(f'  Internet Svc   : {internet_mode}')
    print(f'  Contract Type  : {contract_mode}')
    print(f'  Churn Rate     : {churn_rate:.1f}%')
    persona_notes[f'Cluster_{cid}'] = {
        'size': size,
        'pct_of_total': round(size / len(df_profile) * 100, 2),
        'avg_tenure_months': round(avg_tenure, 2),
        'avg_monthly_charges_usd': round(avg_monthly, 2),
        'dominant_internet_service': internet_mode,
        'dominant_contract_type': contract_mode,
        'churn_rate_pct': round(churn_rate, 2)
    }

---

## 10. Model and Artifact Persistence

All modeling artifacts are serialized to `clustering_knowledge/` for production readiness, audit trails, and future inference on new data.

| Artifact | Format | Description |
|---|---|---|
| `kmeans_k4_model.pkl` | Pickle (joblib) | Trained K-Means model |
| `standard_scaler.pkl` | Pickle (joblib) | Fitted StandardScaler |
| `pca_2d.pkl` | Pickle (joblib) | Fitted PCA (2D) for visualization |
| `feature_names.json` | JSON | Feature names used in modeling |
| `cluster_profiles.csv` | CSV | Numeric cluster summary table |
| `cluster_personas.json` | JSON | Business-level persona metadata |
| `df_with_clusters.csv` | CSV | Full dataset with cluster assignments |
| `model_metadata.json` | JSON | Training parameters and evaluation metrics |
| `k_selection_metrics.csv` | CSV | K-range metric comparison table |

In [ ]:
joblib.dump(kmeans_final, os.path.join(KNOWLEDGE_DIR, 'kmeans_k4_model.pkl'))
joblib.dump(scaler,        os.path.join(KNOWLEDGE_DIR, 'standard_scaler.pkl'))
joblib.dump(pca_2d,        os.path.join(KNOWLEDGE_DIR, 'pca_2d.pkl'))

feature_names = df_encoded.columns.tolist()
with open(os.path.join(KNOWLEDGE_DIR, 'feature_names.json'), 'w') as f:
    json.dump(feature_names, f, indent=2)

cluster_summary.to_csv(os.path.join(KNOWLEDGE_DIR, 'cluster_profiles.csv'))

with open(os.path.join(KNOWLEDGE_DIR, 'cluster_personas.json'), 'w') as f:
    json.dump(persona_notes, f, indent=2)

df.to_csv(os.path.join(KNOWLEDGE_DIR, 'df_with_clusters.csv'), index=False)

model_metadata = {
    'algorithm': 'K-Means',
    'optimal_k': OPTIMAL_K,
    'init_method': 'k-means++',
    'n_init': 50,
    'max_iter': 500,
    'random_state': 42,
    'n_features_used': X_scaled.shape[1],
    'n_samples': X_scaled.shape[0],
    'n_iterations_to_converge': int(kmeans_final.n_iter_),
    'final_inertia': round(float(kmeans_final.inertia_), 4),
    'evaluation_metrics': {
        'silhouette_score': round(float(final_silhouette), 4),
        'davies_bouldin_index': round(float(final_db_score), 4),
        'calinski_harabasz_score': round(float(final_ch_score), 4)
    },
    'features_excluded': FEATURES_TO_DROP,
    'encoding_strategy': {
        'binary': '0/1 direct mapping',
        'service_ordinal': '0=No internet service, 1=No, 2=Yes',
        'contract': '0=Month-to-month, 1=One year, 2=Two year',
        'internet_service': 'One-Hot Encoding',
        'payment_method': 'One-Hot Encoding'
    },
    'scaling': 'StandardScaler (zero mean, unit variance)',
    'dimensionality_reduction': 'PCA 2D/3D for visualization only'
}

with open(os.path.join(KNOWLEDGE_DIR, 'model_metadata.json'), 'w') as f:
    json.dump(model_metadata, f, indent=2)

metrics_df.to_csv(os.path.join(KNOWLEDGE_DIR, 'k_selection_metrics.csv'), index=False)

print('All artifacts saved to:', os.path.abspath(KNOWLEDGE_DIR))
print('\nSaved files:')
for fname in sorted(os.listdir(KNOWLEDGE_DIR)):
    fsize = os.path.getsize(os.path.join(KNOWLEDGE_DIR, fname))
    print(f'  {fname:<45} {fsize:>10,} bytes')

---

## 11. Evaluation Summary

This section consolidates the complete evaluation of the clustering model across all relevant dimensions: quantitative internal validity metrics, cluster size distribution, and post-hoc business validation via churn rate overlays.

In [ ]:
print('=' * 65)
print('   CUSTOMER SEGMENTATION - FINAL EVALUATION REPORT')
print('=' * 65)
print()
print('ALGORITHM       : K-Means Clustering')
print(f'OPTIMAL K       : {OPTIMAL_K}')
print(f'DATASET SIZE    : {X_scaled.shape[0]:,} customers')
print(f'FEATURE SPACE   : {X_scaled.shape[1]} features (after encoding + scaling)')
print(f'CONVERGENCE     : {kmeans_final.n_iter_} iterations')
print(f'FINAL INERTIA   : {kmeans_final.inertia_:,.2f}')
print()
print('EVALUATION METRICS')
print('-' * 40)
print(f'  Silhouette Score         : {final_silhouette:.4f}  (range: -1 to +1, higher is better)')
print(f'  Davies-Bouldin Index     : {final_db_score:.4f}  (lower is better)')
print(f'  Calinski-Harabasz Score  : {final_ch_score:.2f}  (higher is better)')
print()
print('CLUSTER SUMMARY')
print('-' * 65)
print(f'{"Cluster":<10}{"Size":<10}{"Share%":<10}{"Avg Tenure":<14}{"Avg Monthly$":<15}{"Churn%":<10}')
print('-' * 65)
for cid in range(OPTIMAL_K):
    subset = df_profile[df_profile['Cluster'] == cid]
    size   = len(subset)
    share  = size / len(df_profile) * 100
    avg_t  = subset['tenure'].mean()
    avg_m  = subset['MonthlyCharges'].mean()
    churn_r = (subset['Churn'] == 'Yes').mean() * 100
    print(f'{cid:<10}{size:<10,}{share:<10.1f}{avg_t:<14.1f}{avg_m:<15.2f}{churn_r:<10.1f}')
print()
print('ARTIFACTS LOCATION')
print('-' * 40)
print(f'  {os.path.abspath(KNOWLEDGE_DIR)}')
print()
print('=' * 65)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Final Summary Dashboard', fontsize=16, fontweight='bold')

sil_norm = final_silhouette
db_inv   = 1 / (1 + final_db_score)
ch_norm  = final_ch_score / max(calinski_harabasz_values)
metric_vals   = [sil_norm, db_inv, ch_norm]
metric_labels = ['Silhouette\nScore', 'Davies-Bouldin\n(Inverted)', 'CH Score\n(Normalized)']
metric_colors = ['#55A868', '#DD8452', '#4C72B0']

bars = axes[0].bar(metric_labels, metric_vals, color=metric_colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, metric_vals):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', fontsize=12, fontweight='bold')
axes[0].set_ylim(0, 1.15)
axes[0].set_title('Normalized Evaluation Metrics (K=4)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Score (0-1 scale)')
axes[0].axhline(0.5, color='gray', linestyle='--', alpha=0.5)

sizes       = [len(df_profile[df_profile['Cluster'] == k]) for k in range(OPTIMAL_K)]
churn_rates = [(df_profile[df_profile['Cluster'] == k]['Churn'] == 'Yes').mean() * 100
               for k in range(OPTIMAL_K)]

bars2 = axes[1].bar([f'Cluster {k}' for k in range(OPTIMAL_K)], churn_rates,
                    color=CLUSTER_COLORS, edgecolor='white', width=0.55)
for bar, cr, sz in zip(bars2, churn_rates, sizes):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.4,
                 f'{cr:.1f}%\n(n={sz:,})', ha='center', fontsize=9, fontweight='bold')
overall_churn = (df_profile['Churn'] == 'Yes').mean() * 100
axes[1].axhline(overall_churn, color='black', linestyle='--',
                linewidth=1.5, label=f'Overall ({overall_churn:.1f}%)')
axes[1].legend(fontsize=10)
axes[1].set_title('Post-hoc Churn Rate by Cluster', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Churn Rate (%)')

plt.tight_layout()
plt.savefig(os.path.join(KNOWLEDGE_DIR, 'final_summary_dashboard.png'), bbox_inches='tight')
plt.show()
print('Saved: final_summary_dashboard.png')

---

## Conclusion

This notebook has delivered a complete, production-grade unsupervised segmentation pipeline for telecom customers. The K-Means model with **K=4** was selected based on the convergence of four independent diagnostic metrics (Elbow, Silhouette, Davies-Bouldin, and Calinski-Harabasz), ensuring the choice is statistically grounded rather than arbitrary.

**Key Takeaways:**

- The clustering successfully identified distinct customer groups based entirely on service usage behavior, contract commitment, and financial spending, without any reliance on the churn label.
- Post-hoc validation by overlaying the actual `Churn` label confirms that clusters exhibit meaningfully different churn propensities, validating the model's ability to discover structurally real segments.
- The segments discovered provide directly actionable intelligence for customer retention strategy, product bundling decisions, and pricing optimization across each customer persona.

**Recommended Next Steps:**

1. Apply the saved model (`kmeans_k4_model.pkl` + `standard_scaler.pkl`) to new incoming customer records for real-time segment assignment.
2. Build a supervised churn prediction model that uses cluster membership as an additional feature to improve prediction precision.
3. Design cluster-specific retention campaigns and A/B test their effectiveness against the baseline churn rate.